# EXACT Full Server + Symbolic Prehandlers + Quick Test

This notebook starts a local FastAPI `/predict` server with:

1. **v40.4 entity-conjunctive prehandler** for real Phase‑1 style MC/entity rules.
2. **v38b+v39 generic prehandler** for quantified-template logic.
3. Existing LoRA/v35/vLLM fallback placeholder/proxy, enabled only if a vLLM server is available.
4. Deterministic Type2 quick handlers for smoke tests.

## Inputs

Required: none for quick tests.

Optional auto-detected:
- `exact_eval_round1_Astatine.json` for Phase‑1 replay quick tests.
- Qwen3-8B base and Type1 LoRA paths if `RUN_VLLM=True`.

## Outputs

- `/kaggle/working/live_v38b_v39_wrapper.py`
- `/kaggle/working/v40_4_entity_conjunctive_engine.py`
- `/kaggle/working/server_quick_test_report.json`
- `/kaggle/working/server_phase1_symbolic_replay_report.json` if Phase‑1 log exists.

Default: `RUN_VLLM=False`, `RUN_TUNNEL=False`. Set later when server setup becomes priority.

In [ ]:
# CELL 0 — config + write symbolic engines
from pathlib import Path
import os, sys, json, glob, re, time, subprocess, textwrap, math
WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/mnt/data')
WORK.mkdir(parents=True, exist_ok=True)
RUN_VLLM = False   # keep false for now; symbolic + server quick tests do not need vLLM
RUN_TUNNEL = False # keep false unless you need public URL
HOST = '127.0.0.1'
PORT = 9000
VLLM_HOST='127.0.0.1'
VLLM_PORT=8001
BASE_MODEL = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'
TYPE1_LORA = '/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1'
LORA_SERVED_NAME = 'qwen3-8b-exact-type1'

(Path(WORK)/'live_v38b_v39_wrapper.py').write_text('# live_v38b_v39_wrapper.py — self-contained symbolic pre-handler (no deps, no LLM).\n\n# v38b engine (unchanged)\n# CELL 1 — v39 canonical predicate\nimport re\n# ---------- v39-lite: canonical predicate ----------\ndef canon_atom(s):\n    s=str(s).strip()\n    s=re.sub(r\'\\(x\\)|\\(\\s*x\\s*\\)\',\'\',s)\n    s=s.strip()\n    # FOL CamelCase atom -> as-is canonical key\n    if re.fullmatch(r\'[A-Za-z][A-Za-z0-9]*\', s):\n        return s\n    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join\n    STOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'his\',\'her\',\'its\',\n          \'all\',\'every\',\'each\',\'some\',\'any\',\'there\',\'is\',\'are\',\'do\',\'does\',\'did\',\'student\',\'students\',\'researcher\',\n          \'researchers\',\'who\',\'which\',\'it\',\'they\',\'them\',\'then\',\'if\',\'not\'}\n    toks=re.findall(r"[a-zA-Z]+", s.lower())\n    out=[]\n    for t in toks:\n        if t in STOP: continue\n        t=re.sub(r\'(ies)$\',\'y\',t); t=re.sub(r\'(es|s)$\',\'\',t); t=re.sub(r\'(ing|ed)$\',\'\',t)\n        out.append(t)\n    return "_".join(out) if out else s.lower()\n\ndef _norm_tokens(text):\n    text=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(text))\n    toks=re.findall(r\'[a-zA-Z]+\', text.lower())\n    STOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'his\',\'her\',\'its\',\n          \'all\',\'every\',\'each\',\'some\',\'any\',\'there\',\'is\',\'are\',\'do\',\'does\',\'did\',\'student\',\'students\',\'researcher\',\n          \'researchers\',\'who\',\'which\',\'it\',\'they\',\'them\',\'then\',\'if\',\'not\',\'one\',\'least\',\'according\',\'premise\',\n          \'premises\',\'following\',\'statement\',\'true\',\'based\',\'above\',\'can\',\'be\',\'inferred\',\'supported\',\'logically\'}\n    def _stem(t):\n        if re.search(r\'(ss|us|is)$\', t): pass\n        elif re.search(r\'(ches|shes|xes|zes|ses)$\', t): t=t[:-2]\n        elif re.search(r\'ies$\', t): t=t[:-3]+\'y\'\n        elif t.endswith(\'s\'): t=t[:-1]\n        t=re.sub(r\'(ing|ed)$\',\'\',t)\n        return t\n    out=set()\n    for t in toks:\n        if t in STOP: continue\n        t=_stem(t)\n        if t: out.add(t)\n    return out\n\n# CELL 2 — FOL parser\n# ---------- FOL parser ----------\ndef parse_fol(fol):\n    """Return (\'rule\',A,B) | (\'uni\',A) | (\'uni_neg\',A) | (\'exist\',A) | (\'exist_neg\',A) | None"""\n    f=str(fol).replace(\'->\',\'→\').replace(\'¬\',\'~\').replace(\'∀\',\'A\').replace(\'∃\',\'E\')\n    f=f.strip()\n    # implication\n    m=re.search(r\'\\(?\\s*([~]?\\s*[A-Za-z0-9]+)\\s*\\(x\\)\\s*→\\s*([~]?\\s*[A-Za-z0-9]+)\\s*\\(x\\)\\s*\\)?\', f)\n    if m and f.startswith(\'A\'):\n        a=m.group(1).replace(\' \',\'\'); b=m.group(2).replace(\' \',\'\')\n        an=a.startswith(\'~\'); bn=b.startswith(\'~\')\n        return (\'rule\', (canon_atom(a.lstrip(\'~\')),an), (canon_atom(b.lstrip(\'~\')),bn))\n    # quantified single atom\n    m=re.search(r\'^([AE])\\s*x?\\s*\\(?\\s*(~?)\\s*([A-Za-z0-9]+)\\s*\\(x\\)\\s*\\)?$\', f)\n    if m:\n        quant,neg,pred=m.group(1),m.group(2)==\'~\',canon_atom(m.group(3))\n        if quant==\'A\': return (\'uni_neg\',pred) if neg else (\'uni\',pred)\n        else: return (\'exist_neg\',pred) if neg else (\'exist\',pred)\n    # ¬∃x P  == ∀¬P\n    m=re.search(r\'~\\s*E\\s*x?\\s*\\(?\\s*([A-Za-z0-9]+)\\s*\\(x\\)\', f)\n    if m: return (\'uni_neg\',canon_atom(m.group(1)))\n    return None\n\n# CELL 3 — Closure\n# ---------- closure ----------\ndef build_closure(premises_fol):\n    rules=[]; uni=set(); uni_neg=set(); exist=set()\n    prov={}  # atom -> premise idx that introduced it (for path)\n    for i,fol in enumerate(premises_fol):\n        p=parse_fol(fol)\n        if not p: continue\n        if p[0]==\'rule\': rules.append((i,p[1],p[2]))\n        elif p[0]==\'uni\': uni.add(p[1]); prov.setdefault((\'pos\',p[1]),[i])\n        elif p[0]==\'uni_neg\': uni_neg.add(p[1]); prov.setdefault((\'neg\',p[1]),[i])\n        elif p[0]==\'exist\': exist.add(p[1]); prov.setdefault((\'ex\',p[1]),[i])\n    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni\n    changed=True\n    while changed:\n        changed=False\n        for i,(a,an),(b,bn) in rules:\n            # positive modus ponens: rule with both positive\n            if not an and not bn and a in uni and b not in uni:\n                uni.add(b); prov[(\'pos\',b)]=prov.get((\'pos\',a),[])+[i]; changed=True\n            # contrapositive: B false, rule A->B => A false\n            if not an and not bn and b in uni_neg and a not in uni_neg:\n                uni_neg.add(a); prov[(\'neg\',a)]=prov.get((\'neg\',b),[])+[i]; changed=True\n    # existential forward: exist A + A->B => exist B ; uni A => exist A\n    for a in list(uni): exist.add(a); prov.setdefault((\'ex\',a),prov.get((\'pos\',a),[]))\n    changed=True\n    while changed:\n        changed=False\n        for i,(a,an),(b,bn) in rules:\n            if not an and not bn and a in exist and b not in exist:\n                exist.add(b); prov[(\'ex\',b)]=prov.get((\'ex\',a),[])+[i]; changed=True\n    return {\'uni\':uni,\'uni_neg\':uni_neg,\'exist\':exist,\'prov\':prov}\n\n# CELL 4 — Query type + target matching\n# ---------- query type + target ----------\ndef query_type(q):\n    ql=str(q).lower()\n    if re.search(r\'\\bat least one\\b|\\bsome\\b|\\bany\\b|\\bthere (is|exists)\\b|does .* one\', ql): return \'existential\'\n    if re.search(r\'\\bdo all\\b|\\bdoes every\\b|\\ball students\\b|\\bevery\\b|\\beach\\b\', ql): return \'universal\'\n    if re.search(r\'is the following statement true|which (statement|option)|can be inferred|is logically supported\', ql): return \'statement\'\n    return \'unknown\'\n\ndef target_atom(q, atoms):\n    qt=_norm_tokens(q)\n    scored=[]\n    for a in atoms:\n        at=_norm_tokens(a)\n        if not at: continue\n        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question\n        scored.append((ov,len(at & qt),a))\n    scored.sort(reverse=True)\n    if not scored: return None\n    top=scored[0]\n    if top[0] < 0.6 or top[1] < 1: return None\n    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous\n    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]\n    if ties: return None\n    return top[2]\n\n# CELL 5 — YNU projection + certificate (UNCHANGED from v38)\n# ---------- projection with v35 convention + certificate ----------\ndef prove(premises_fol, question):\n    cl=build_closure(premises_fol)\n    atoms=cl[\'uni\']|cl[\'uni_neg\']|cl[\'exist\']|{a for _,(a,_),(b,_) in [] }\n    allatoms=set()\n    for fol in premises_fol:\n        p=parse_fol(fol)\n        if not p: continue\n        if p[0]==\'rule\': allatoms.add(p[1][0]); allatoms.add(p[2][0])\n        else: allatoms.add(p[1])\n    qt=query_type(question); tgt=target_atom(question, allatoms)\n    cert={\'query_type\':qt,\'target\':tgt,\'positive\':None,\'negative\':None,\'answer\':None,\'premises_used\':[],\'abstain_reason\':None}\n    if tgt is None:\n        cert[\'answer\']=None; cert[\'abstain_reason\']=\'target_not_matched\'; return cert\n    pos = tgt in cl[\'uni\'] or tgt in cl[\'exist\']\n    neg = tgt in cl[\'uni_neg\']\n    cert[\'positive\']=pos; cert[\'negative\']=neg\n    if qt==\'existential\':\n        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)\n            cert[\'answer\']=\'No\'; cert[\'premises_used\']=cl[\'prov\'].get((\'neg\',tgt),[]); cert[\'proof_rule\']=\'E1_universal_negative\'\n        elif pos:\n            cert[\'answer\']=\'Yes\'; cert[\'premises_used\']=cl[\'prov\'].get((\'ex\',tgt),cl[\'prov\'].get((\'pos\',tgt),[])); cert[\'proof_rule\']=\'PE_witness\'\n        else:\n            cert[\'answer\']=None; cert[\'abstain_reason\']=\'no_proof\'\n    elif qt==\'universal\':\n        if tgt in cl[\'uni\']:  # PY: positive universal wins\n            cert[\'answer\']=\'Yes\'; cert[\'premises_used\']=cl[\'prov\'].get((\'pos\',tgt),[]); cert[\'proof_rule\']=\'PY_universal_positive\'\n        elif neg:\n            cert[\'answer\']=\'No\'; cert[\'premises_used\']=cl[\'prov\'].get((\'neg\',tgt),[]); cert[\'proof_rule\']=\'U1_universal_negative\'\n        else:\n            cert[\'answer\']=None; cert[\'abstain_reason\']=\'no_proof\'\n    else:\n        cert[\'answer\']=None; cert[\'abstain_reason\']=\'statement_or_mc_out_of_scope\'\n    cert[\'premises_used\']=sorted(set(cert[\'premises_used\']))\n    return cert\n\n# CELL 6 — v38b MC: option-type classifier + conditional-distractor exclusion + meta policy\ndef classify_option(opt):\n    t=re.sub(r"^\\s*[A-Da-d][.):]\\s*","",str(opt).strip())  # strip "A." / "B)" prefix if present\n    tl=t.lower()\n    if re.search(r"cannot be (determined|inferred)|undetermined|does not (support|allow)|no conclusion|insufficient", tl):\n        return "META"\n    # conditional / relative-clause distractor: "X who/that ... must/will/should ..." or "if ... then ..."\n    if re.search(r"\\bwho\\b|\\bthat\\b", tl) and re.search(r"\\b(must|will|should|then)\\b", tl): return "CONDITIONAL"\n    if re.search(r"^\\s*if\\b", tl): return "CONDITIONAL"\n    if re.search(r"\\bmust\\b", tl): return "CONDITIONAL"   # malformed "must completes"\n    if re.search(r"^\\s*no\\b", tl): return "UNIV_NEG"\n    if re.search(r"^\\s*(only some|some only)\\b", tl): return "PARTIAL"\n    if re.search(r"^\\s*(at least one|some|there (is|exists))\\b", tl): return "EXIST_POS"\n    if re.search(r"^\\s*(every|all|each)\\b", tl): return "UNIV_POS"\n    return "UNKNOWN_OPT"\n\ndef allatoms_of(fol):\n    A=set()\n    for f in fol:\n        p=parse_fol(f)\n        if not p: continue\n        if p[0]=="rule": A.add(p[1][0]); A.add(p[2][0])\n        else: A.add(p[1])\n    return A\n\ndef eval_direct(kind, opt, cl, allatoms):\n    atom=target_atom(opt, allatoms)\n    if atom is None: return "UNSUPPORTED",None\n    if kind=="UNIV_POS": return ("PROVEN" if atom in cl[\'uni\'] else ("DISPROVEN" if atom in cl[\'uni_neg\'] else "UNSUPPORTED")),atom\n    if kind=="UNIV_NEG": return ("PROVEN" if atom in cl[\'uni_neg\'] else ("DISPROVEN" if atom in cl[\'uni\'] else "UNSUPPORTED")),atom\n    if kind=="EXIST_POS": return ("PROVEN" if atom in cl[\'exist\'] else ("DISPROVEN" if atom in cl[\'uni_neg\'] else "UNSUPPORTED")),atom\n    if kind=="PARTIAL":\n        if atom in cl[\'exist\'] and atom not in cl[\'uni\'] and atom not in cl[\'uni_neg\']: return "PROVEN",atom\n        return ("DISPROVEN" if (atom in cl[\'uni\'] or atom in cl[\'uni_neg\']) else "UNSUPPORTED"),atom\n    return "UNSUPPORTED",atom\n\ndef prove_mc_v38b(fol, options):\n    cl=build_closure(fol); allatoms=allatoms_of(fol)\n    labels=list("ABCD")[:len(options)]\n    proven=[]; meta=None; prov_atom=None\n    for lab,opt in zip(labels,options):\n        k=classify_option(opt)\n        if k=="META": meta=lab; continue\n        if k in ("CONDITIONAL","UNKNOWN_OPT"): continue   # never selectable\n        st,atom=eval_direct(k,opt,cl,allatoms)\n        if st=="PROVEN": proven.append((lab,atom))\n    cert={\'answer\':None,\'rule\':None,\'premises_used\':[],\'abstain_reason\':None}\n    if len(proven)==1:\n        lab,atom=proven[0]; cert[\'answer\']=lab; cert[\'rule\']=\'MC_unique_direct_proof\'\n        for key in [(\'pos\',atom),(\'neg\',atom),(\'ex\',atom)]:\n            if key in cl[\'prov\']: cert[\'premises_used\']=sorted(set(cl[\'prov\'][key])); break\n    elif len(proven)==0 and meta is not None:\n        cert[\'answer\']=meta; cert[\'rule\']=\'MC_meta_cannot_determine\'\n    else:\n        cert[\'abstain_reason\']=(\'multiple_direct_proven\' if proven else \'no_direct_and_no_meta\')\n    return cert\nprint(\'v38b MC policy ready\')\n\n# v39 parser\n# -*- coding: utf-8 -*-\n"""v39 NL->predicate parser: NL premises -> canonical FOL atoms, so the v38b engine\nruns in live (NL-only) setting. Atom key = CamelCase of sorted normalized content tokens,\nso NL phrases and the FOL oracle map to the SAME atom namespace -> directly comparable."""\nimport re\nSTOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'his\',\'her\',\'its\',\n      \'all\',\'every\',\'each\',\'some\',\'any\',\'there\',\'is\',\'are\',\'do\',\'does\',\'did\',\'student\',\'students\',\'researcher\',\n      \'researchers\',\'intern\',\'interns\',\'developer\',\'developers\',\'employee\',\'employees\',\'candidate\',\'candidates\',\n      \'member\',\'members\',\'person\',\'people\',\'who\',\'which\',\'it\',\'they\',\'them\',\'then\',\'if\',\'one\',\'least\',\'must\',\n      \'will\',\'should\'}  # domain nouns kept (course/exam carry meaning) intentionally\n\ndef _stem(t):\n    if re.search(r\'(ss|us|is)$\',t): return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\',t): return t[:-2]\n    if re.search(r\'ies$\',t): return t[:-3]+\'y\'\n    if t.endswith(\'s\'): t=t[:-1]\n    return re.sub(r\'(ing|ed)$\',\'\',t)\ndef _toks(text):\n    text=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(text))\n    out=[]\n    for w in re.findall(r\'[a-zA-Z]+\',text.lower()):\n        if w in STOP: continue\n        s=_stem(w)\n        if s: out.append(s)\n    return out\ndef atom_key(phrase):\n    t=sorted(set(_toks(phrase)))\n    return "".join(w.capitalize() for w in t) if t else None\n\nNEG=re.compile(r\'\\b(no|not|never|cannot|can\\\'t|does not|do not|doesn\\\'t|don\\\'t|fails? to|unable)\\b\',re.I)\ndef _polarity(s): return bool(NEG.search(s))\n\ndef nl_premise_to_fol(nl):\n    s=str(nl).strip()\n    m=re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\',s,re.I)\n    if m:\n        a,b=m.group(1),m.group(2)\n        ak,bk=atom_key(a),atom_key(b)\n        if not ak or not bk: return None\n        an=\'¬\' if _polarity(a) else \'\'; bn=\'¬\' if _polarity(b) else \'\'\n        return f\'∀x ({an}{ak}(x) → {bn}{bk}(x))\'\n    m=re.search(r\'^\\s*(every|all|each)\\b(.+)$\',s,re.I)\n    if m:\n        body=m.group(2); k=atom_key(body)\n        if not k: return None\n        return f\'∀x ({"¬" if _polarity(body) else ""}{k}(x))\'\n    m=re.search(r\'^\\s*no\\b(.+)$\',s,re.I)\n    if m:\n        k=atom_key(m.group(1));  return f\'∀x (¬{k}(x))\' if k else None\n    m=re.search(r\'^\\s*(at least one|some|there (?:is|exists)|at least)\\b(.+)$\',s,re.I)\n    if m:\n        body=m.group(2); k=atom_key(body)\n        if not k: return None\n        return f\'∃x ({"¬" if _polarity(body) else ""}{k}(x))\'\n    return None\n\ndef nl_to_canon(premises_nl):\n    return [nl_premise_to_fol(p) for p in premises_nl]\n\ndef fol_to_canon(premises_fol):\n    """Re-emit oracle FOL into the SAME atom namespace (CamelCase of sorted tokens)."""\n    out=[]\n    for f in premises_fol:\n        s=str(f)\n        def rep(m):\n            neg=m.group(1) or \'\'\n            k=atom_key(m.group(2))\n            return f\'{neg}{k}(x)\'\n        s2=re.sub(r\'(¬?)\\s*([A-Za-z][A-Za-z0-9]*)\\(x\\)\',rep,s)\n        out.append(s2)\n    return out\n\n# LIVE wrapper: run v38b certificate engine on NL-only premises (no FOL available)\nimport re\ndef parse_opts(q): return [o[1].strip().replace("\\n"," ") for o in re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)", q, flags=re.S)]\ndef _is_ynu_options(options):\n    vals={str(o).strip().lower() for o in (options or [])}; vals={v for v in vals if v}\n    return bool(vals) and vals <= {"yes","no","unknown","uncertain"}\n\ndef verify_v38_live(question, premises_nl, options=None):\n    """NL premises -> canonical FOL -> v38b proof. Returns (answer|None, premises_used, rule), cert."""\n    canon=nl_to_canon(premises_nl)\n    if not any(canon):\n        return (None,[],"nl_parse_empty"), {"answer":None,"abstain_reason":"nl_parse_empty","canon_premises":canon}\n    opts=options or parse_opts(question)\n    # MC when options are real answer choices (not Yes/No/Uncertain)\n    if opts and not _is_ynu_options(opts):\n        c=prove_mc_v38b(canon, opts)\n        c["canon_premises"]=canon\n        return (c.get("answer"), c.get("premises_used",[]), c.get("rule") or c.get("abstain_reason")), c\n    c=prove(canon, question)\n    c["canon_premises"]=canon\n    return (c.get("answer"), c.get("premises_used",[]), c.get("proof_rule") or c.get("abstain_reason")), c\n\nprint("verify_v38_live ready (NL-only)")\n\n# ================= UNIT TESTS (run: python live_v38b_v39_wrapper.py) =================\nMC_OPTS=["A. Every student completes the coursework.",\n         "B. It cannot be determined whether every student completes the coursework.",\n         "C. No student completes the coursework.",\n         "D. Every student who earns course credit must completes."]\nMC_Q="Which statement is correct?\\n"+"\\n".join(MC_OPTS)\n\ndef _run_unit_tests():\n    res=[]\n    def t(name, q, premises, options, exp, exp_prem=None):\n        (a,pu,why),_=verify_v38_live(q, premises, options)\n        ok=(a==exp) and (exp_prem is None or sorted(pu)==sorted(exp_prem))\n        res.append((ok,name,exp,a,why,pu))\n    chain=["Every researcher reads the literature.",\n           "If a researcher reads the literature, then the researcher identifies a gap.",\n           "If a researcher identifies a gap, then the researcher designs a study."]\n    t("YNU positive chain -> Yes",     "Do all researchers design a study?", chain, None, "Yes", [0,1,2])\n    eneg=["Every researcher improves technique.",\n          "If a researcher improves technique, then the researcher scores above threshold.",\n          "No researcher scores above threshold."]\n    t("YNU existential-negative -> No","Does at least one researcher improve technique?", eneg, None, "No")\n    t("Unknown / no proof -> abstain", "Do all students submit all assignments?",\n      ["If a student submits all assignments, then the student achieves a high GPA.","Every student achieves a high GPA."], None, None)\n    mc_prov=["If a student earns course credit, then the student completes the coursework.",\n             "Every student earns course credit."]\n    t("MC unique direct -> A",  MC_Q, mc_prov, MC_OPTS, "A")\n    mc_meta=["If a student earns course credit, then the student completes the coursework."]  # A/C unprovable\n    t("MC meta cannot-determine -> B", MC_Q, mc_meta, MC_OPTS, "B")\n    t("MC conditional distractor must NOT win -> B", MC_Q, mc_meta, MC_OPTS, "B")\n    npass=sum(1 for r in res if r[0])\n    print(f"UNIT TESTS: {npass}/{len(res)} passed")\n    for ok,name,exp,got,why,pu in res:\n        print(("  PASS " if ok else "  FAIL ")+f"{name}: exp={exp} got={got} rule={why} prem={pu}")\n    return npass==len(res)\n\nif __name__=="__main__":\n    import sys\n    ok=_run_unit_tests()\n    (a,pu,why),_=verify_v38_live("Does at least one student receive a scholarship?",\n                                 ["Every student receives a scholarship."], ["Yes","No","Uncertain"])\n    print("\\nlive-format smoke (NL-only, no FOL):", a, pu, why)\n    sys.exit(0 if ok else 1)\n', encoding='utf-8')
(Path(WORK)/'v40_4_entity_conjunctive_engine.py').write_text('# -*- coding: utf-8 -*-\n"""v40.3 entity-grounded conjunctive Horn engine for the REAL Phase-1 distribution.\nPropositional over a single entity: facts = literals, rules = (conj of literals)->literal.\nForward-chain; answer options by derivability. Certificate = premise indices. Abstain-safe."""\nimport re\nSTOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'its\',\'it\',\'they\',\'them\',\n      \'is\',\'are\',\'was\',\'were\',\'be\',\'been\',\'has\',\'have\',\'had\',\'then\',\'if\',\'no\',\'not\',\'with\',\'as\',\'by\',\'from\',\n      \'artifact\',\'package\',\'manuscript\',\'sample\',\'batch\',\'item\',\'device\',\'record\',\'file\',\'student\'}\ndef _stem(t):\n    if re.search(r\'(ss|us|is)$\',t): return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\',t): return t[:-2]\n    if re.search(r\'ies$\',t): return t[:-3]+\'y\'\n    if t.endswith(\'s\'): t=t[:-1]\n    return re.sub(r\'(ing|ed)$\',\'\',t)\ndef atom_key(phrase):\n    s=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(phrase)).lower()\n    nums=re.findall(r\'\\d+\', s)\n    toks=[ _stem(w) for w in re.findall(r\'[a-zA-Z]+\', s) ]\n    toks=[t for t in toks if t and t not in STOP and len(t)>2]\n    keys=sorted(set(toks))+["N"+n for n in sorted(set(nums))]   # keep numeric literals\n    return "".join(w.capitalize() for w in keys) if keys else None\n\n# split a clause into (atom, neg). Handles "X has Y", "X is Y", "X has no Y", "X lacks Y", "cannot ...", "is not ..."\n_LEAD=re.compile(r"^\\s*(if|then|that|who|which|it|its|their|this)\\b",re.I)\n_VERB=re.compile(r"\\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn\'t|aren\'t|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\\b",re.I)\nACTION_VERBS={\'receives\',\'receive\',\'provides\',\'provide\',\'shows\',\'show\',\'states\',\'state\',\'monitors\',\'monitor\',\'captures\',\'capture\',\'enters\',\'enter\',\'requires\',\'require\',\'needs\',\'need\',\'gains\',\'gain\',\'completed\',\'complete\',\'contains\',\'contain\',\'reports\',\'report\',\'releases\',\'release\',\'passes\',\'pass\',\'improves\',\'improve\',\'supports\',\'support\',\'recommends\',\'recommend\',\'administered\',\'administer\',\'approved\',\'approve\'}\n_NEGWORD=re.compile(r\'\\b(no|not|cannot|can not|lacks?|without|isn\\\'t|aren\\\'t|never|nor|incomplete|missing|lacking)\\b\',re.I)\ndef to_literal(clause):\n    c=clause.strip().rstrip(\'.\').strip()\n    c=_LEAD.sub(\'\',c).strip()\n    neg=bool(re.search(r"\\b(no|not|cannot|can not|never|lacks?|without|isn\'t|aren\'t|incomplete|missing|lacking|nor|un(?:able|verified|established))\\b",c,re.I))\n    m=_VERB.search(c)\n    pred=c[m.end():].strip() if m else c\n    verb=(m.group(1).lower() if m else \'\')\n    if m and verb in ACTION_VERBS:\n        pred = verb + \' \' + pred\n    # peel any leftover leading modal/aux/passive markers and articles\n    for _ in range(4):\n        pred=re.sub(r"^\\s*(be|been|being|to|a|an|the|no|not|its|their)\\b","",pred,flags=re.I).strip()\n    a=atom_key(pred)\n    return (a,neg) if a else None\n\ndef parse_premise(p):\n    s=str(p).strip()\n    m=re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\',s,re.I)\n    if m:\n        ante=re.split(r\'\\band\\b\',m.group(1),flags=re.I)\n        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]\n        con=to_literal(m.group(2))\n        if con and lits: return (\'rule\',lits,con)\n        return None\n    # Universal relative rule: Every/All <role> <condition> <verb> <consequent>\n    # Examples: Every volunteer assigned to shift receives badge; All satellites with cameras can capture images.\n    m2=re.search(r\'^\\s*(every|all)\\s+[a-zA-Z]+s?\\s+(.+?)\\s+\\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\\b\\s+(.+)$\',s,re.I)\n    if m2:\n        cond=m2.group(2).strip()\n        cons=(m2.group(3)+" "+m2.group(4)).strip()\n        litc=to_literal(cond); litd=to_literal(cons)\n        if litc and litd: return (\'rule\',[litc],litd)\n    if re.search(r\'^\\s*(no premise|it (is|cannot)|unknown|there is no information)\',s,re.I): return None\n    lit=to_literal(s)\n    return (\'fact\',lit) if lit else None\n\ndef solve_entity(premises):\n    facts={}  # atom -> (bool_value, premise_idx)\n    rules=[]\n    for i,p in enumerate(premises):\n        pp=parse_premise(p)\n        if not pp: continue\n        if pp[0]==\'fact\':\n            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))\n        else:\n            rules.append((i,pp[1],pp[2]))\n    changed=True\n    while changed:\n        changed=False\n        for i,lits,con in rules:\n            ca,cneg=con\n            ok=True; path=[i]\n            for a,neg in lits:\n                if a in facts and facts[a][0]==(not neg): path+=facts[a][1]\n                else: ok=False; break\n            if ok and ca not in facts:\n                facts[ca]=((not cneg),sorted(set(path))); changed=True\n    return facts\n\n_META_RE=__import__("re").compile(r"\\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\\b",__import__("re").I)\ndef decompose_option(opt):\n    import re\n    t=re.sub(r\'^\\s*[A-Da-d][.):]\\s*\',\'\',str(opt)).strip()\n    t=re.split(r\'\\bbecause\\b\', t, maxsplit=1, flags=re.I)[0].strip()  # drop causal justification\n    parts=re.split(r\',\\s*but\\s+|\\s+but\\s+|;\\s+|\\s+while\\s+|\\s+whereas\\s+|\\s+and\\s+\',t,flags=re.I)\n    claims=[]\n    for p in parts:\n        p=p.strip()\n        if not p: continue\n        is_meta=bool(_META_RE.search(p))\n        lit=to_literal(p)\n        claims.append((lit,is_meta,p))\n    return claims\n\ndef answer_mc(premises, options):\n    facts=solve_entity(premises)\n    res={}\n    for lab,opt in zip("ABCD",options):\n        claims=decompose_option(opt)\n        if not claims: res[lab]=(\'UNSUP\',[]); continue\n        status=\'PROVEN\'; path=[]\n        for lit,is_meta,txt in claims:\n            if lit is None: status=\'UNSUP\'; break\n            a,neg=lit; have = a in facts\n            val = facts[a][0] if have else None\n            if is_meta:\n                # meta "not established": correct only if NOT positively proven\n                if have and val==True: status=\'DISPROVEN\'; break\n            else:\n                if have and val==(not neg): path+=facts[a][1]\n                elif have and val==(neg): status=\'DISPROVEN\'; break\n                else: status=\'UNSUP\'; break\n        res[lab]=(status, sorted(set(path)))\n    proven=[l for l in res if res[l][0]==\'PROVEN\']\n    if len(proven)==1: return proven[0],res[proven[0]][1],\'entity_unique_proof\',res\n    return None,[],(\'multiple\' if proven else \'none\'),res\n\n\n# ================= Phase-1 REPLAY HARNESS =================\ndef _opt_texts(rp):\n    import re\n    f=[o[1].strip().replace("\\n"," ") for o in re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)",rp.get("query",""),flags=re.S)]\n    return f if len(f)>=2 else (rp.get("options") or [])\ndef replay_phase1(path):\n    import json,re\n    d=json.load(open(path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]\n    fired=aok=pok=0; fixed=[]; wrong=[]; abst=0\n    for l in t1:\n        rp=l["request_payload"]; exp=l.get("expected") or {}; ea=str(exp.get("answer","")).strip().upper()\n        opts=_opt_texts(rp)\n        if not opts: continue\n        a,pu,why,res=answer_mc(rp.get("premises",[]) or [], opts)\n        if a is None: abst+=1; continue\n        fired+=1; ok=(a==ea); aok+=ok; pok+=(sorted(pu)==sorted(exp.get("premises_used") or []))\n        if ok and l.get("status")!="correct": fixed.append(l["query_id"])\n        if not ok: wrong.append({"query_id":l["query_id"],"exp":ea,"got":a})\n    rep={"n":len(t1),"fired":fired,"answer_correct":aok,"premises_used_correct":pok,\n         "wrong":wrong,"fixed_old_wrong":fixed,"abstained":abst,\n         "precision_when_fired":round(aok/max(fired,1),3),"coverage":round(fired/max(len(t1),1),3),\n         "gate":"ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY"}\n    return rep\ndef _autofind():\n    import glob,os,sys\n    if len(sys.argv)>1 and os.path.exists(sys.argv[1]): return sys.argv[1]\n    for c in ["exact_eval_round1_Astatine.json","/kaggle/input/**/exact_eval_round1_Astatine.json",\n              "/kaggle/working/exact_eval_round1_Astatine.json","./exact_eval_round1_Astatine.json"]:\n        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])\n        if h: return h[0]\n    return None\nif __name__=="__main__":\n    import json\n    p=_autofind()\n    if not p: print("exact_eval_round1_Astatine.json not found; pass path as arg."); raise SystemExit(1)\n    rep=replay_phase1(p); json.dump(rep,open("v40_phase1_replay_report.json","w"),indent=1)\n    print(json.dumps(rep,indent=1))\n', encoding='utf-8')
sys.path.insert(0, str(WORK))
print('Wrote engines into', WORK)

In [ ]:
# CELL 1 — import engines and run local unit smoke
from live_v38b_v39_wrapper import verify_v38_live
import v40_4_entity_conjunctive_engine as v40
print('Imported symbolic engines')

# v38/v39 smoke
(a, pu, why), cert = verify_v38_live(
    'Does at least one student receive a scholarship?',
    ['Every student receives a scholarship.'],
    ['Yes','No','Uncertain']
)
print('v38/v39 smoke:', a, pu, why)
assert a == 'Yes'

# v40 smoke using Phase1 style Amber Amulet
amber_premises = [
 'If an artifact has a humidity-control log and no pest-damage report, then it is storage-ready.',
 'If an artifact is storage-ready and has a provenance certificate, then it is eligible for exhibition.',
 'If an artifact is eligible for exhibition and has curator approval, then it can be placed on public display.',
 'If an artifact is fragile, then it requires low-light protection.',
 'If an artifact requires low-light protection and can be placed on public display, then it must be displayed in a climate-controlled case.',
 'The Amber Amulet has a humidity-control log.',
 'The Amber Amulet has no pest-damage report.',
 'The Amber Amulet has a provenance certificate.',
 'The Amber Amulet has curator approval.',
 'The Amber Amulet is fragile.',
]
amber_options = [
 'The Amber Amulet must be displayed in a climate-controlled case',
 'The Amber Amulet cannot be placed on public display',
 'The Amber Amulet needs pest treatment before storage',
 'The Amber Amulet lacks a provenance certificate',
]
a,pu,why,res = v40.answer_mc(amber_premises, amber_options)
print('v40 smoke:', a, pu, why)
assert a == 'A' and sorted(pu)==list(range(10))
print('Symbolic unit smoke PASS')

In [ ]:
# CELL 2 — optional vLLM server health/proxy helpers; does not launch by default
import requests, json, time, os, subprocess, shlex, sys, pathlib

def vllm_models_ok(timeout=3):
    try:
        r = requests.get(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/models', timeout=timeout)
        return r.status_code == 200, r.text[:500]
    except Exception as e:
        return False, repr(e)

def vllm_completion(prompt, max_tokens=128):
    payload = {'model': LORA_SERVED_NAME, 'prompt': prompt, 'temperature':0.0, 'max_tokens':max_tokens}
    r = requests.post(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/completions', json=payload, timeout=60)
    r.raise_for_status()
    return r.json()['choices'][0].get('text','')

ok,msg = vllm_models_ok()
print('Existing vLLM:', ok, msg[:200])
if RUN_VLLM and not ok:
    print('RUN_VLLM=True, but launcher is intentionally not auto-started in this compact server notebook.')
    print('Use the previous vLLM waitfix launcher if you need /v1/models compliance.')

In [ ]:
# CELL 3 — Type1 + Type2 handlers
import re, math, json, time, requests

def _field(req, name, default=None):
    if isinstance(req, dict):
        return req.get(name, default)
    return getattr(req, name, default)

def parse_final_answer(text):
    m = re.search(r'Final Answer\s*:\s*([^\n]+)', str(text), flags=re.I)
    if m:
        return m.group(1).strip().split()[0].strip(' .')
    if re.search(r'cannot (?:be )?(?:determined|inferred)|unknown|uncertain', str(text), flags=re.I):
        return 'Unknown'
    return 'Unknown'

def parse_options_from_query(q):
    return [o[1].strip().replace('\n',' ') for o in re.findall(r'(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)', str(q), flags=re.S)]

def symbolic_type1(req):
    qid = _field(req, 'query_id')
    q = _field(req, 'query', '') or ''
    premises = list(_field(req, 'premises', []) or [])
    options = list(_field(req, 'options', []) or [])

    # v40 first for real Phase1-style MC/entity/conjunctive cases
    opt_texts = parse_options_from_query(q) if options and all(str(o).strip().upper() in list('ABCD') for o in options) else options
    if opt_texts and not {str(o).strip().lower() for o in opt_texts} <= {'yes','no','unknown','uncertain'}:
        a, pu, why, res = v40.answer_mc(premises, opt_texts)
        if a is not None:
            return {
                'query_id': qid, 'answer': a, 'unit': '',
                'explanation': f'Derived by v40.4 entity-conjunctive symbolic proof ({why}).',
                'premises_used': pu,
                'reasoning': {'source':'v40_4_entity_conjunctive', 'rule':why, 'option_status':res}
            }

    # v38/v39 generic quantified-template fallback
    (sa, sp, why), cert = verify_v38_live(q, premises, options)
    if sa is not None:
        ans = sa
        if ans == 'Unknown' and any(str(o).strip().lower() == 'uncertain' for o in options):
            ans = 'Uncertain'
        return {
            'query_id': qid, 'answer': ans, 'unit': '',
            'explanation': f'Derived by v38b/v39 symbolic proof ({why}).',
            'premises_used': sp,
            'reasoning': {'source':'v38b_v39_symbolic', 'rule':why, 'canon_premises': cert.get('canon_premises', [])}
        }
    return None

def lora_fallback_type1(req):
    qid = _field(req,'query_id')
    ok,_ = vllm_models_ok(timeout=2)
    if ok:
        # Minimal fallback prompt; deploy notebook can replace with locked artifact prompt.
        q = _field(req,'query','') or ''
        premises = _field(req,'premises',[]) or []
        options = _field(req,'options',[]) or []
        prompt = 'Use only the given premises.\nPremises:\n' + '\n'.join(f'{i}. {p}' for i,p in enumerate(premises)) + f'\nQuestion: {q}\nOptions: {options}\nEnd with one line: Final Answer: X\nReasoning:\n'
        try:
            raw = vllm_completion(prompt, max_tokens=256)
            ans = parse_final_answer(raw)
        except Exception as e:
            raw = f'[vllm_error] {e!r}'
            ans = 'Unknown'
    else:
        raw = '[no_vllm_available]'
        ans = 'Unknown'
    return {'query_id': qid, 'answer': ans, 'unit':'', 'explanation':'Fallback Type1 answer.', 'premises_used':[], 'reasoning': {'source':'lora_fallback', 'raw': raw[:500]}}

def predict_type1(req):
    sym = symbolic_type1(req)
    if sym is not None:
        return sym
    return lora_fallback_type1(req)

def _num(pattern, text, default=None):
    m = re.search(pattern, text, flags=re.I)
    return float(m.group(1)) if m else default

def predict_type2(req):
    qid = _field(req,'query_id')
    q = _field(req,'query','') or ''
    text = q.replace('μ','u').replace('Ω','ohm')
    # Parallel resistors total current: R1, R2, V
    if re.search(r'parallel', text, re.I) and re.search(r'resistor', text, re.I):
        nums = [float(x) for x in re.findall(r'R\s*\d*\s*=\s*([0-9.]+)', text, flags=re.I)]
        V = _num(r'(?:V|U|voltage|battery|potential difference)\s*=\s*([0-9.]+)', text)
        if V is None:
            m=re.search(r'across a\s*([0-9.]+)\s*V', text, flags=re.I); V=float(m.group(1)) if m else None
        if len(nums)>=2 and V is not None:
            I = sum(V/r for r in nums if r)
            return {'query_id':qid,'answer': f'{I:g}', 'unit':'A', 'explanation':'Total current in parallel is the sum of branch currents I=V/R.', 'premises_used':[], 'reasoning': {'source':'type2_parallel_resistor','R':nums,'V':V}}
    # Capacitor energy: E=1/2 C U^2, with uF -> J/mJ
    if re.search(r'capacit', text, re.I) and re.search(r'energy|stored', text, re.I):
        C = _num(r'C\s*=\s*([0-9.]+)', text)
        U = _num(r'(?:U|V|voltage|potential difference)\s*=\s*([0-9.]+)', text)
        if C is not None and U is not None:
            C_F = C*1e-6 if re.search(r'uF|micro\s*F', text, re.I) else C
            E = 0.5*C_F*U*U
            # Return mJ if convenient like previous quick accepted
            if E < 0.1:
                return {'query_id':qid,'answer': f'{E*1000:.4g}', 'unit':'mJ', 'explanation':'Capacitor energy is 1/2 C U^2.', 'premises_used':[], 'reasoning': {'source':'type2_capacitor_energy','C_F':C_F,'U':U}}
            return {'query_id':qid,'answer': f'{E:.4g}', 'unit':'J', 'explanation':'Capacitor energy is 1/2 C U^2.', 'premises_used':[], 'reasoning': {'source':'type2_capacitor_energy','C_F':C_F,'U':U}}
    return {'query_id':qid,'answer':'', 'unit':'', 'explanation':'Type2 fallback: unsupported physics family.', 'premises_used':[], 'reasoning': {'source':'type2_fallback'}}

def predict_one(req):
    typ = (_field(req,'type','') or '').lower()
    if typ == 'type2': return predict_type2(req)
    return predict_type1(req)

In [ ]:
# CELL 4 — FastAPI server
import threading, uvicorn, nest_asyncio, time, requests, json
try:
    import fastapi
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'nest_asyncio'])
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
nest_asyncio.apply()
app = FastAPI()

@app.get('/health')
def health():
    ok,msg = vllm_models_ok(timeout=1)
    return {'ok': True, 'symbolic': True, 'vllm_models_ok': ok, 'vllm_msg': msg[:200]}

@app.get('/v1/models')
def proxy_models():
    ok,msg = vllm_models_ok(timeout=2)
    if ok:
        try:
            r=requests.get(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/models', timeout=5)
            return JSONResponse(r.json(), status_code=r.status_code)
        except Exception as e:
            return JSONResponse({'error':repr(e)}, status_code=502)
    return JSONResponse({'object':'list','data':[], 'warning':'vLLM not running; symbolic quick-test server only'}, status_code=503)

@app.post('/v1/completions')
async def proxy_completions(request: Request):
    try:
        payload = await request.json()
        r=requests.post(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/completions', json=payload, timeout=120)
        return JSONResponse(r.json(), status_code=r.status_code)
    except Exception as e:
        return JSONResponse({'error':repr(e), 'warning':'vLLM not running or failed'}, status_code=503)

@app.post('/predict')
async def predict(request: Request):
    payload = await request.json()
    items = payload if isinstance(payload, list) else [payload]
    out = [predict_one(x) for x in items]
    return out

_server = None

def start_server():
    global _server
    config = uvicorn.Config(app, host=HOST, port=PORT, log_level='warning')
    _server = uvicorn.Server(config)
    thread = threading.Thread(target=_server.run, daemon=True)
    thread.start()
    for _ in range(40):
        try:
            r=requests.get(f'http://{HOST}:{PORT}/health', timeout=2)
            if r.status_code==200:
                print('Server ready:', r.json())
                return
        except Exception:
            time.sleep(0.5)
    raise RuntimeError('Server did not start')

start_server()

In [ ]:
# CELL 5 — Quick API tests
import requests, json, time
base=f'http://{HOST}:{PORT}'

T1_AMBER = {
 'query_id':'T1_amber', 'type':'type1',
 'query':'Based on the museum conservation rules, which conclusion is logically supported?\nA. The Amber Amulet must be displayed in a climate-controlled case\nB. The Amber Amulet cannot be placed on public display\nC. The Amber Amulet needs pest treatment before storage\nD. The Amber Amulet lacks a provenance certificate',
 'premises':[
  'If an artifact has a humidity-control log and no pest-damage report, then it is storage-ready.',
  'If an artifact is storage-ready and has a provenance certificate, then it is eligible for exhibition.',
  'If an artifact is eligible for exhibition and has curator approval, then it can be placed on public display.',
  'If an artifact is fragile, then it requires low-light protection.',
  'If an artifact requires low-light protection and can be placed on public display, then it must be displayed in a climate-controlled case.',
  'The Amber Amulet has a humidity-control log.',
  'The Amber Amulet has no pest-damage report.',
  'The Amber Amulet has a provenance certificate.',
  'The Amber Amulet has curator approval.',
  'The Amber Amulet is fragile.'
 ],
 'options':['A','B','C','D']
}
T1_YNU = {'query_id':'T1_ynu','type':'type1','query':'Does at least one student receive a scholarship?','premises':['Every student receives a scholarship.'],'options':['Yes','No','Uncertain']}
T2_PARALLEL = {'query_id':'T2_parallel','type':'type2','query':'Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.','premises':[],'options':[]}
T2_CAP = {'query_id':'T2_cap','type':'type2','query':'A capacitor has capacitance C = 47 uF and is connected to a potential difference U = 12 V. Calculate the energy stored in the capacitor.','premises':[],'options':[]}

tests=[
    (T1_AMBER, {'answer':'A', 'premises_used': list(range(10))}),
    (T1_YNU, {'answer':'Yes', 'premises_used': [0]}),
    (T2_PARALLEL, {'answer':'5', 'unit':'A'}),
    (T2_CAP, {'unit':'mJ'}),
]
report=[]
for payload, expected in tests:
    r=requests.post(base+'/predict', json=payload, timeout=30)
    print('\n', payload['query_id'], r.status_code, r.text[:500])
    obj=r.json()[0]
    ok=True
    if 'answer' in expected: ok = ok and str(obj.get('answer'))==str(expected['answer'])
    if 'unit' in expected: ok = ok and str(obj.get('unit'))==str(expected['unit'])
    if 'premises_used' in expected: ok = ok and sorted(obj.get('premises_used') or [])==sorted(expected['premises_used'])
    report.append({'query_id':payload['query_id'], 'ok': ok, 'response': obj, 'expected': expected})

OUT = WORK/'server_quick_test_report.json'
OUT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('\nSaved:', OUT)
print('PASS', sum(x['ok'] for x in report), '/', len(report))
assert all(x['ok'] for x in report), report

In [ ]:
# CELL 6 — Optional Phase-1 symbolic replay if exact_eval_round1_Astatine.json is available
import glob, os, json, re
cands=[]
for pat in ['/kaggle/working/**/exact_eval_round1_Astatine.json','/kaggle/input/**/exact_eval_round1_Astatine.json','/mnt/data/**/exact_eval_round1_Astatine.json','./exact_eval_round1_Astatine.json']:
    cands += glob.glob(pat, recursive=True)
cands=sorted(set([p for p in cands if os.path.exists(p)]))
print('Phase1 candidates:', cands[:5])
if cands:
    rep=v40.replay_phase1(cands[0])
    OUT=WORK/'server_phase1_symbolic_replay_report.json'
    OUT.write_text(json.dumps(rep, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(rep, indent=2, ensure_ascii=False))
    print('Saved:', OUT)
else:
    print('No Phase1 log found; skip replay.')

## Notes

- This notebook is for quick integration and smoke testing. It intentionally keeps `RUN_VLLM=False` by default.
- For official compliance, run your proven vLLM server separately and verify `/v1/models` before public submission.
- Symbolic-fired Type1 requests should return instantly; abstained requests fall back to LoRA/v35 when vLLM is available.